# Install libraries

In [1]:
# Installation
!pip install --quiet --upgrade pandas==2.2.2 numpy==2.0.0
!pip install --quiet --upgrade langgraph yfinance ta textblob langchain langchain-core pydantic nest-asyncio gtts gradio
!pip install --quiet langgraph-checkpoint-sqlite

# Imports libraries

In [2]:
# COMPLETE CORRECTED CODE (with Custom Serializer)

import asyncio
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
from typing import Dict, List, Optional, Literal, Tuple, Any
from enum import Enum
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict, Annotated
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.serde.base import SerializerProtocol
import pickle
from textblob import TextBlob
from ta.momentum import StochasticOscillator, WilliamsRIndicator
from ta.trend import SMAIndicator
from ta.volume import OnBalanceVolumeIndicator
import gradio as gr
from gtts import gTTS
import warnings
warnings.filterwarnings('ignore') # Filter out warnings that are not critical for execution

# Summary

1. Yahoo Finance (yfinance) se real-time ya historical stock data le raha hai.

2. Technical Indicators (ta library) - jaise Stochastic Oscillator, Williams %R, SMA, On Balance Volume - use karke trend aur momentum analyze karta hai.

3. Sentiment Analysis (TextBlob) - news ya social media text ka positive/negative emotion nikalta hai.

4. LangGraph - ek state machine / workflow banata hai jisme multiple steps hote hain (e.g., data fetch - analyze - decide - notify).

5. Gradio - ek simple web UI provide karta hai jaha user stock symbol daal kar result dekh sakta hai.

6. gTTS (Google Text-to-Speech) - analysis result ko voice me suna sakta hai.

7. Custom Serializer + Pickle - LangGraph ke checkpoint memory me state save karne ke liye, taake process interrupt ho jaaye to wahi se restart ho.

In [3]:
# CUSTOM SERIALIZER TO HANDLE PANDAS DATAFRAME

class CustomSerializer(SerializerProtocol):
    """Handles pandas DataFrame serialization for LangGraph checkpointer"""

    def dumps(self, obj: Any) -> bytes:
        """Serialize object to bytes"""
        if isinstance(obj, pd.DataFrame):
            # Convert DataFrame to dict and then pickle
            return pickle.dumps(("DataFrame", obj.to_dict()))
        return pickle.dumps(obj)

    def dumps_typed(self, obj: Any) -> Tuple[str, bytes]:
        """Return type string and serialized bytes"""
        if isinstance(obj, pd.DataFrame):
            return "DataFrame", pickle.dumps(obj.to_dict())
        return "pickle", pickle.dumps(obj)

    def loads(self, data: bytes) -> Any:
        """Deserialize bytes to object"""
        obj = pickle.loads(data)
        if isinstance(obj, tuple) and len(obj) == 2 and obj[0] == "DataFrame":
            return pd.DataFrame.from_dict(obj[1])
        return obj

    def loads_typed(self, data: Tuple[str, bytes]) -> Any:
        """Deserialize typed data"""
        type_str, bytes_data = data
        if type_str == "DataFrame":
            return pd.DataFrame.from_dict(pickle.loads(bytes_data))
        return pickle.loads(bytes_data)

Ye code pandas DataFrame ko safely save aur load karne ke liye hai, khaas taur par LangGraph checkpointer ke andar.
Short mein:

1. dumps - Object ko bytes mein convert karta hai. Agar object DataFrame hai, toh pehle usse dict mein badalta hai, phir pickle karta hai (saath me ("DataFrame", dict) tag laga kar).

2. dumps_typed - Same logic, par type bhi batata hai ("DataFrame" ya "pickle").

3. loads - Bytes se original object banaata hai. Agar tag "DataFrame" mila, toh dict se wapas DataFrame banata hai.

4. loads_typed - Type ke hisaab se deserialize karta hai.

Simple words:
DataFrame ko seedha pickle nahi karta (kyunki kabhi kabhi version issues aate hain), pehle to_dict() se todta hai, phir pickle karta hai, aur load karte waqt wapas DataFrame bana deta hai. Isse LangGraph ke checkpoint system me DataFrame reliably store ho jaati hai.

In [4]:
# SECTION 1: PYDANTIC SCHEMAS (Structured Output)

class RiskLevel(str, Enum):
    CRITICAL = "CRITICAL"
    HIGH = "HIGH"
    MODERATE = "MODERATE"
    LOW = "LOW"
    NEGLIGIBLE = "NEGLIGIBLE"

class TechSignal(BaseModel):
    primary_direction: str = Field(description="Bullish/Bearish/Neutral")
    momentum_state: str = Field(description="Rising/Falling/Flat")
    overbought_oversold_status: str = Field(description="Overbought/Oversold/Neutral")
    key_support_resistance: Dict[str, float] = Field(default_factory=dict)
    hidden_divergence: str = Field(description="Present/Absent")
    composite_score: float = Field(ge=-1.0, le=1.0)
    conviction: RiskLevel = Field(default=RiskLevel.MODERATE)

class SentimentState(BaseModel):
    crowd_mood: str = Field(description="Fear/Greed/Neutral/Extreme")
    news_volume_trend: str = Field(description="Surge/Normal/Decline")
    unusual_activity_detected: bool = Field(default=False)
    main_keywords: List[str] = Field(default_factory=list)
    earnings_forecast: str = Field(description="Beat/Miss/Inline/Unknown")
    polarity_score: float = Field(ge=-1.0, le=1.0)
    conviction: RiskLevel = Field(default=RiskLevel.MODERATE)

class RiskMetrics(BaseModel):
    var_breach_likelihood: float = Field(ge=0.0, le=1.0)
    min_acceptable_sharpe: float = Field(default=1.0)
    max_allocation_pct: float = Field(ge=0.0, le=1.0)
    suggested_stop_loss: float = Field(ge=0.0, le=0.5)
    tail_risk_present: bool = Field(default=False)
    risk_appetite: str = Field(description="Aggressive/Moderate/Conservative")
    risk_score: float = Field(ge=-1.0, le=1.0)

class VolumeProfile(BaseModel):
    accumulation_phase: str = Field(description="Accumulation/Distribution/Neutral")
    institutional_interest: str = Field(description="High/Medium/Low")
    liquidity_quality: str = Field(description="Deep/Thin/Volatile")
    volume_price_divergence: bool = Field(default=False)
    volume_score: float = Field(ge=-1.0, le=1.0)

class FinalDecision(BaseModel):
    verdict: Literal["STRONG_BUY", "BUY", "HOLD", "SELL", "STRONG_SELL"]
    confidence_band: tuple = Field(description="(low, high)")
    risk_adjusted_rating: float = Field(ge=-2.0, le=2.0)
    investment_horizon: str = Field(description="Short/Mid/Long term")
    explanation_chain: List[str] = Field(default_factory=list)
    conflict_notes: List[str] = Field(default_factory=list)

class AgentDynamicWeights(BaseModel):
    tech_weight: float
    sentiment_weight: float
    risk_weight: float
    volume_weight: float
    last_update: datetime = Field(default_factory=datetime.now)

**Ye code:**

Ek trading/investment analysis system ka structured output (data format) define kar raha hai using Pydantic schemas. Matlab, jo bhi AI ya model output dega, wo is format me hoga — predictable aur type-safe.

Har schema ka matlab:

RiskLevel - Enum for risk (CRITICAL se NEGLIGIBLE tak).

TechSignal - Technical analysis data: direction (Bullish/Bearish), momentum, overbought/oversold, support/resistance levels, divergence, score, aur conviction.

SentimentState - Sentiment analysis: crowd mood (Fear/Greed), news volume, unusual activity, keywords, earnings forecast, polarity score, conviction.

RiskMetrics - Risk related: VaR breach likelihood, Sharpe ratio, max allocation, stop loss, tail risk, risk appetite, risk score.

VolumeProfile - Volume analysis: accumulation/distribution, institutional interest, liquidity, volume-price divergence, volume score.

FinalDecision - Final verdict (STRONG_BUY to STRONG_SELL), confidence band, risk-adjusted rating, investment horizon, explanation chain, conflict notes.

AgentDynamicWeights - Dynamic weights for each module (tech, sentiment, risk, volume) with timestamp.

**Short me bolu toh:**

Ye ek multi-agent trading system ka response format hai, jo technical, sentiment, risk, aur volume data collect karta hai, sabko weights dekar final BUY/SELL/HOLD decision banata hai.

In [5]:
# SECTION 2: LANGGRAPH STATE DEFINITION

class AnalystState(TypedDict):
    messages: Annotated[List[Dict], add_messages]
    ticker: str
    lookback: str
    price_data: Optional[pd.DataFrame]
    tech_agent_raw: Optional[Dict]
    sent_agent_raw: Optional[Dict]
    risk_agent_raw: Optional[Dict]
    vol_agent_raw: Optional[Dict]
    tech_signal_obj: Optional[TechSignal]
    sent_state_obj: Optional[SentimentState]
    risk_metrics_obj: Optional[RiskMetrics]
    vol_profile_obj: Optional[VolumeProfile]
    final_decision_obj: Optional[FinalDecision]
    live_weights: Optional[AgentDynamicWeights]
    error_list: List[str]
    step_timestamps: Dict[str, str]
    consensus_flag: bool

Ye code LangGraph ke state (memory) ko define kiya hai.

**Short mein:**

messages - user/assistant ke conversation ka record (automatically add hota rehta hai).

ticker - stock ya crypto ka symbol (jaise RELIANCE.NS).

lookback - kitne din pichla data dekhna hai.

price_data - price ka DataFrame (agar fetch ho gaya to).

tech_agent_raw, sent_agent_raw, risk_agent_raw, vol_agent_raw - har agent ka raw output (JSON ya dict).

tech_signal_obj, sent_state_obj, risk_metrics_obj, vol_profile_obj - parsed/processed objects jo agents nikaalte hain.

final_decision_obj - final decision (buy/sell/hold) after combining sab agents ka output.

live_weights - dynamically adjust hone wale weights (kaunsa agent kitna important).

error_list - errors store karne ke liye

step_timestamps - har step ka time track karne ke liye

consensus_flag - True / False, ye batata hai ki agents ka majority consensus bana ya nahi

**Ek line mein:** Ye woh notebook hai jo agents (technical, sentiment, risk, volume) ko run karta hai, unke results collect karta hai, aur final trading decision banata hai - sab data ek jagah state mein rakhta hai.

In [6]:
# SECTION 3: DATA FETCHER (Yahoo Finance)

def fetch_stock_data(symbol: str, period: str = "6mo", interval: str = "1d") -> pd.DataFrame:
    """Fetch historical price data from Yahoo Finance"""
    ticker = yf.Ticker(symbol)
    df = ticker.history(period=period, interval=interval)
    if df.empty:
        raise ValueError(f"No data found for symbol {symbol}")
    df.dropna(inplace=True)
    return df

**Ye code Yahoo Finance se kisi stock ka historical data fetch karta hai.**

In simple words :

yf.Ticker(symbol) - stock symbol (jaise "RELIANCE.NS") ke liye ticker object banata hai.

.history(period, interval) - us stock ki price history laata hai (default: last 6 months, daily data).

Agar data nahi mila - ValueError throw karega.

df.dropna() - missing values hata deta hai.

Last me DataFrame return karta hai.

In [7]:
# SECTION 4: TECHNICAL ANALYSIS (Original logic)

def compute_technical_indicators(df: pd.DataFrame) -> Dict:
    """Calculate all technical indicators from scratch"""
    close = df['Close']
    high = df['High']
    low = df['Low']
    volume = df['Volume']

    # Stochastic %K (14,3)
    stoch = StochasticOscillator(high=high, low=low, close=close, window=14, smooth_window=3)
    stoch_k = stoch.stoch().iloc[-1]

    # Williams %R (14)
    williams = WilliamsRIndicator(high=high, low=low, close=close, lbp=14)
    williams_r = williams.williams_r().iloc[-1]

    # SMAs
    sma10 = SMAIndicator(close=close, window=10).sma_indicator().iloc[-1]
    sma30 = SMAIndicator(close=close, window=30).sma_indicator().iloc[-1]

    current_price = close.iloc[-1]

    # Composite score based on trend and momentum
    trend_score = 0.0
    if sma10 > sma30:
        trend_score += 0.3
    elif sma10 < sma30:
        trend_score -= 0.3

    if stoch_k > 80:
        trend_score -= 0.2  # overbought
    elif stoch_k < 20:
        trend_score += 0.2  # oversold

    if williams_r > -20:
        trend_score -= 0.2
    elif williams_r < -80:
        trend_score += 0.2

    # Clamp between -1 and 1
    tech_score = max(-1.0, min(1.0, trend_score))

    # Signal mapping
    if tech_score > 0.4:
        signal = "BUY"
    elif tech_score < -0.4:
        signal = "SELL"
    else:
        signal = "NEUTRAL"

    return {
        "tech_score": tech_score,
        "tech_signal": signal,
        "stoch_k": round(stoch_k, 2),
        "williams_r": round(williams_r, 2),
        "price": round(current_price, 2),
        "sma_10": round(sma10, 2),
        "sma_30": round(sma30, 2)
    }


**Ye code technical analysis kar raha hai, jo stock ya crypto ke price data se indicators calculate karta hai aur BUY/SELL/NEUTRAL signal decide karta hai.**

**Short me kya ho raha hai:**

Indicators calculate:

Stochastic %K (14 period) - batata hai market overbought (>80) ya oversold (<20) hai.

Williams %R (14 period) - same concept, overbought > -20, oversold < -80.

SMA 10 aur SMA 30 - short-term aur long-term average.

Trend score banta hai (starting 0):

Agar SMA10 > SMA30 → +0.3 (uptrend), warna -0.3 (downtrend).

Agar Stochastic overbought (>80) → -0.2 (sell signal), oversold (<20) - +0.2 (buy signal).

Agar Williams overbought (> -20) → -0.2, oversold (< -80) → +0.2.

Final tech_score -1 se +1 ke beech clamp hota hai.

Signal mapping:

Score > 0.4 - BUY

Score < -0.4 - SELL

Bich me - NEUTRAL

Return hota hai: score, signal, or sab indicators ki values.

In simple words: Code price trend aur momentum dekh kar decide karta hai ki ab khareedna hai, bechna hai, ya wait karna hai.

In [8]:
# SECTION 5: SENTIMENT ANALYSIS (Mock news + TextBlob)

def fetch_sentiment_for_symbol(symbol: str) -> Dict:
    """
    Simulate news sentiment using TextBlob on a few dummy headlines.
    In production, you can replace with NewsAPI or Twitter.
    """
    # Dummy headlines related to the symbol (just for demo)
    headlines = [
        f"{symbol} launches new product line, analysts optimistic",
        f"Concerns over {symbol} supply chain disruptions",
        f"{symbol} earnings expected to beat estimates",
        f"Regulatory headwinds may affect {symbol}",
        f"Strong retail interest in {symbol} stock"
    ]
    polarities = [TextBlob(h).sentiment.polarity for h in headlines]
    avg_polarity = np.mean(polarities)
    buzz_level = min(1.0, len(headlines) / 10.0)  # mock buzz

    # Verdict
    if avg_polarity > 0.2:
        verdict = "BULLISH"
    elif avg_polarity < -0.2:
        verdict = "BEARISH"
    else:
        verdict = "NEUTRAL"

    confidence = min(1.0, abs(avg_polarity) * 2)
    # Simulate geo threat (random for demo, but deterministic)
    geo_threat = max(0.0, min(1.0, abs(avg_polarity) * 0.5 + 0.1))

    return {
        "verdict": verdict,
        "confidence": round(confidence, 2),
        "polarity": round(avg_polarity, 3),
        "buzz_level": round(buzz_level, 2),
        "geo_threat": round(geo_threat, 2)
    }

**Ye code kisi stock symbol (jaise RELIANCE, AAPL) ke liye fake news headlines banata hai aur unka sentiment (bhaavna) analyze karta hai using TextBlob library.**

Short me kahe to:

Function ek symbol lega (e.g., "TESLA").

Us symbol ke naam se 5 dummy headlines banayega (kuch positive, kuch negative).

Har headline ka polarity score (-1 se +1) nikalega (positive = acchi khabar, negative = buri).

Inka average polarity calculate karega.

Is average ke hisaab se verdict dega:

0.2 - BULLISH (positive)

< -0.2 - BEARISH (negative)

else - NEUTRAL

Saath me confidence, buzz level (fake), aur geo_threat (fake) bhi nikal dega.

Simple words: Ye news headlines ka mood check karta hai aur batata hai ki market up jaayega ya down - lekin sab kuch mock (nakli) data hai, real news nahi.

In [9]:
# SECTION 6: RISK METRICS COMPUTATION

def calculate_risk_indicators(df: pd.DataFrame) -> Dict:
    """Compute VaR, CVaR, volatility, drawdown from scratch"""
    returns = df['Close'].pct_change().dropna()
    volatility = returns.std() * np.sqrt(252) * 100  # annualized %
    # VaR 95% (historical)
    var_95 = np.percentile(returns, 5) * 100  # in percentage
    # CVaR 95%
    cvar_95 = returns[returns <= np.percentile(returns, 5)].mean() * 100
    # Max Drawdown
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_dd = drawdown.min() * 100
    # Sortino ratio (risk-free rate = 0 for simplicity)
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std() if len(downside_returns) > 0 else 0.01
    sortino = (returns.mean() * 252) / (downside_std * np.sqrt(252)) if downside_std != 0 else 0
    # Calmar = annual return / max drawdown
    annual_return = returns.mean() * 252
    calmar = annual_return / abs(max_dd/100) if max_dd != 0 else 0

    # Risk grade
    if volatility < 20 and abs(max_dd) < 25:
        grade = "LOW"
    elif volatility > 40 or abs(max_dd) > 40:
        grade = "HIGH"
    else:
        grade = "MEDIUM"

    return {
        "Volatility_annual": round(volatility, 2),
        "VaR_95": round(var_95, 2),
        "CVaR_95": round(cvar_95, 2),
        "Max_Drawdown": round(max_dd, 2),
        "Sortino": round(sortino, 2),
        "Calmar": round(calmar, 2),
        "risk_grade": grade
    }


**Ye code ek stock/index ke risk indicators calculate kar raha hai. **

**Short me:**

Volatility - Returns ka annualized standard deviation (risk measure).

VaR 95% - 5% worst-case loss (historical percentile).

CVaR 95% - Average loss un extreme 5% cases me.

Max Drawdown - Sabse badi fall (peak se bottom tak).

Sortino ratio - Risk-adjusted return (sirf downside risk dekhta hai).

Calmar ratio - Annual return ÷ max drawdown.

Risk Grade - LOW/MEDIUM/HIGH based on volatility aur drawdown.

Code 100% sahi hai agar:

Input df me Close column ho.

Returns normal ho (no NaN issues).

Risk-free rate 0 maan liya ho (Sortino me).

Overall logic sahi hai. Bas thoda sa edge case handle hai (downside_std=0, max_dd=0).

In [10]:
# SECTION 7: VOLUME PATTERN ANALYSIS

def analyze_volume_patterns(df: pd.DataFrame) -> Dict:
    """Volume ratio, OBV, price-volume correlation"""
    volume = df['Volume']
    close = df['Close']

    # Volume ratio (5-day avg / 20-day avg)
    vol_5 = volume.rolling(5).mean().iloc[-1]
    vol_20 = volume.rolling(20).mean().iloc[-1]
    vol_ratio = vol_5 / vol_20 if vol_20 != 0 else 1.0

    # OBV momentum
    obv_indicator = OnBalanceVolumeIndicator(close=close, volume=volume)
    obv = obv_indicator.on_balance_volume()
    obv_momentum = obv.diff().iloc[-5:].mean()

    # Price-volume correlation (last 30 days)
    price_returns = close.pct_change().dropna()
    vol_returns = volume.pct_change().dropna()
    if len(price_returns) > 1 and len(vol_returns) > 1:
        corr = price_returns.corr(vol_returns)
    else:
        corr = 0

    # Signal
    if vol_ratio > 1.2 and obv_momentum > 0:
        signal = "ACCUMULATION"
    elif vol_ratio < 0.8 and obv_momentum < 0:
        signal = "DISTRIBUTION"
    else:
        signal = "NEUTRAL"

    return {
        "vol_signal": signal,
        "vol_ratio": round(vol_ratio, 2),
        "price_vol_corr": round(corr, 2),
        "obv_momentum": round(obv_momentum, 0)
    }

**Ye code volume patterns analyze kar raha hai — matlab dekh raha hai ki stock ya asset mein volume kaise behave kar raha hai price ke saath.**

Short mein samjhao:

Volume ratio - Last 5 din ka average volume / last 20 din ka average volume. Agar >1.2 hai matlab recent volume zyada hai.

OBV momentum - On Balance Volume indicator ka last 5 din ka average change. Positive matlab buying pressure, negative matlab selling pressure.

Price-volume correlation - Last 30 din mein price returns aur volume returns ke beech correlation. Positive matlab dono same direction mein move karte hain.

Final signal:

ACCUMULATION - jab volume ratio >1.2 (volume badh raha) aur OBV momentum positive (buying pressure) - matlab smart money accumulate kar rahi hai.

DISTRIBUTION - jab volume ratio <0.8 (volume kam ho raha) aur OBV momentum negative - matlab selling ho rahi hai.

NEUTRAL - baaki conditions mein.

**Ek line mein:**

Volume ka recent trend, OBV momentum, aur price-volume correlation dekh ke bata raha hai ki accumulation ho rahi hai, distribution, ya neutral.

In [11]:
# SECTION 8: LANGGRAPH AGENT NODES (Wrapped)

async def technical_expert_node(state: AnalystState) -> Dict:
    try:
        df = state["price_data"] if state.get("price_data") is not None else fetch_stock_data(state["ticker"], state["lookback"])
        raw = compute_technical_indicators(df)
        tech_score = raw["tech_score"]
        primary_dir = "Bullish" if tech_score > 0.2 else "Bearish" if tech_score < -0.2 else "Neutral"
        momentum = "Rising" if raw["stoch_k"] > 70 else "Falling" if raw["stoch_k"] < 30 else "Flat"
        oos = "Overbought" if raw["stoch_k"] > 80 else "Oversold" if raw["stoch_k"] < 20 else "Neutral"

        tech_obj = TechSignal(
            primary_direction=primary_dir,
            momentum_state=momentum,
            overbought_oversold_status=oos,
            key_support_resistance={"sma_10": raw["sma_10"], "sma_30": raw["sma_30"]},
            hidden_divergence="Absent",
            composite_score=tech_score,
            conviction=RiskLevel.HIGH if abs(tech_score) > 0.6 else RiskLevel.MODERATE
        )
        return {
            "tech_agent_raw": raw,
            "tech_signal_obj": tech_obj,
            "step_timestamps": {**state.get("step_timestamps", {}), "technical": datetime.now().isoformat()}
        }
    except Exception as e:
        return {"error_list": state.get("error_list", []) + [f"Tech agent error: {str(e)}"]}

async def sentiment_expert_node(state: AnalystState) -> Dict:
    try:
        raw = fetch_sentiment_for_symbol(state["ticker"])
        mood = "Greed" if raw["polarity"] > 0.2 else "Fear" if raw["polarity"] < -0.2 else "Neutral"
        sent_obj = SentimentState(
            crowd_mood=mood,
            news_volume_trend="Surge" if raw["buzz_level"] > 0.6 else "Normal",
            unusual_activity_detected=raw["buzz_level"] > 0.8,
            main_keywords=["earnings", "outlook"] if "beat" in raw["verdict"].lower() else ["uncertainty"],
            earnings_forecast="Beat" if raw["polarity"] > 0.1 else "Miss" if raw["polarity"] < -0.1 else "Inline",
            polarity_score=raw["polarity"],
            conviction=RiskLevel.LOW if raw["confidence"] < 0.4 else RiskLevel.MODERATE
        )
        return {
            "sent_agent_raw": raw,
            "sent_state_obj": sent_obj,
            "step_timestamps": {**state.get("step_timestamps", {}), "sentiment": datetime.now().isoformat()}
        }
    except Exception as e:
        return {"error_list": state.get("error_list", []) + [f"Sentiment agent error: {str(e)}"]}

async def risk_expert_node(state: AnalystState) -> Dict:
    try:
        df = state["price_data"] if state.get("price_data") is not None else fetch_stock_data(state["ticker"], state["lookback"])
        raw = calculate_risk_indicators(df)
        var_prob = 0.05 if raw["Volatility_annual"] < 20 else 0.15
        risk_obj = RiskMetrics(
            var_breach_likelihood=var_prob,
            min_acceptable_sharpe=1.0,
            max_allocation_pct=0.25 if raw["risk_grade"] == "LOW" else 0.10,
            suggested_stop_loss=0.12 if raw["risk_grade"] == "LOW" else 0.07,
            tail_risk_present=raw["Volatility_annual"] > 40,
            risk_appetite="Aggressive" if raw["risk_grade"] == "LOW" else "Conservative" if raw["risk_grade"] == "HIGH" else "Moderate",
            risk_score=-0.5 if raw["risk_grade"] == "HIGH" else 0.5 if raw["risk_grade"] == "LOW" else 0.0
        )
        return {
            "risk_agent_raw": raw,
            "risk_metrics_obj": risk_obj,
            "step_timestamps": {**state.get("step_timestamps", {}), "risk": datetime.now().isoformat()}
        }
    except Exception as e:
        return {"error_list": state.get("error_list", []) + [f"Risk agent error: {str(e)}"]}

async def volume_expert_node(state: AnalystState) -> Dict:
    try:
        df = state["price_data"] if state.get("price_data") is not None else fetch_stock_data(state["ticker"], state["lookback"])
        raw = analyze_volume_patterns(df)
        acc_phase = "Accumulation" if raw["vol_signal"] == "ACCUMULATION" else "Distribution" if raw["vol_signal"] == "DISTRIBUTION" else "Neutral"
        vol_obj = VolumeProfile(
            accumulation_phase=acc_phase,
            institutional_interest="High" if raw["vol_ratio"] > 1.2 else "Medium",
            liquidity_quality="Deep" if raw["vol_ratio"] > 0.8 else "Thin",
            volume_price_divergence=abs(raw["price_vol_corr"]) < 0.2,
            volume_score=0.4 if raw["vol_signal"] == "ACCUMULATION" else -0.4 if raw["vol_signal"] == "DISTRIBUTION" else 0.0
        )
        return {
            "vol_agent_raw": raw,
            "vol_profile_obj": vol_obj,
            "step_timestamps": {**state.get("step_timestamps", {}), "volume": datetime.now().isoformat()}
        }
    except Exception as e:
        return {"error_list": state.get("error_list", []) + [f"Volume agent error: {str(e)}"]}

**Ye code:**

LangGraph ke 4 alag-alag "expert" nodes hain, jo ek stock (ticker) ko analyze karte hain:

Technical Expert - Price data se technical indicators nikalta hai (SMA, Stochastics, etc.), fir ek score banata hai Bullish/Bearish/Neutral aur momentum batata hai.

Sentiment Expert - News ya social media sentiment fetch karta hai, Greed/Fear/Neutral mood detect karta hai, aur keywords nikalta hai.

Risk Expert - Volatility, Sharpe ratio, stop-loss suggest karta hai. Risk grade (Low/High) decide karta hai aur allocation % batata hai.

Volume Expert - Volume patterns dekhta hai (Accumulation ya Distribution), institutional interest aur liquidity assess karta hai.

Common baat:

Har node state leta hai (jo pehle se ticker, price_data rakhta hai).

Agar data missing hai toh khud fetch kar leta hai (fetch_stock_data).

Analysis kar ke ek object (jaise TechSignal) banata hai.

State mein update karta hai (tech_signal_obj, sent_state_obj, etc.) aur timestamp add karta hai.

Agar koi error aaya, toh error_list mein entry daal deta hai.

Short mein: Ye 4 agents parallel ya sequential ho sakte hain, har apne angle se stock analyze karta hai, aur final decision ke liye data collect karta hai.

In [12]:
# SECTION 9: SUPERVISOR CROSS-VALIDATION

async def supervisor_node(state: AnalystState) -> Dict:
    tech_obj = state.get("tech_signal_obj")
    sent_obj = state.get("sent_state_obj")
    risk_obj = state.get("risk_metrics_obj")
    vol_obj = state.get("vol_profile_obj")

    explanation = []
    conflicts = []

    tech_score = tech_obj.composite_score if tech_obj else 0
    sent_score = sent_obj.polarity_score if sent_obj else 0
    risk_score = risk_obj.risk_score if risk_obj else 0
    vol_score = vol_obj.volume_score if vol_obj else 0

    # Detect conflicts
    signs = [np.sign(tech_score), np.sign(sent_score), np.sign(risk_score), np.sign(vol_score)]
    if sum(1 for s in signs if s > 0) == 3 and sum(1 for s in signs if s < 0) == 1:
        conflicts.append("Three agents bullish, one bearish – majority rules")
    elif sum(1 for s in signs if s < 0) == 3 and sum(1 for s in signs if s > 0) == 1:
        conflicts.append("Three agents bearish, one bullish – caution advised")
    if abs(tech_score - sent_score) > 0.6:
        conflicts.append(f"Tech ({tech_score:.2f}) vs Sentiment ({sent_score:.2f}) divergence")

    # Dynamic weighting
    weights = AgentDynamicWeights(tech_weight=0.35, sentiment_weight=0.20, risk_weight=0.25, volume_weight=0.20)
    if risk_score < -0.3:
        weights.risk_weight = 0.45
        weights.tech_weight = 0.30
        weights.sentiment_weight = 0.15
        weights.volume_weight = 0.10
        explanation.append("High risk environment → increased risk weight")
    if abs(sent_score) > 0.6:
        weights.sentiment_weight = 0.35
        weights.tech_weight = 0.30
        weights.risk_weight = 0.20
        weights.volume_weight = 0.15
        explanation.append("Extreme sentiment → sentiment weight boosted")

    total = sum([weights.tech_weight, weights.sentiment_weight, weights.risk_weight, weights.volume_weight])
    norm_tech = weights.tech_weight / total
    norm_sent = weights.sentiment_weight / total
    norm_risk = weights.risk_weight / total
    norm_vol = weights.volume_weight / total

    composite = (tech_score * norm_tech + sent_score * norm_sent +
                 risk_score * norm_risk + vol_score * norm_vol)

    # Final call
    if composite > 0.6: verdict = "STRONG_BUY"
    elif composite > 0.25: verdict = "BUY"
    elif composite > -0.25: verdict = "HOLD"
    elif composite > -0.6: verdict = "SELL"
    else: verdict = "STRONG_SELL"

    # Confidence band
    variance = np.var([tech_score, sent_score, risk_score, vol_score])
    margin = 0.15 * (1 + variance)
    low = max(-1.0, composite - margin)
    high = min(1.0, composite + margin)

    risk_adj = composite * (1 - len(conflicts) * 0.1)

    # Horizon
    if abs(tech_score) > abs(sent_score) and abs(tech_score) > abs(risk_score):
        horizon = "Short-term"
        explanation.append("Technical dominant → short-term focus")
    elif abs(sent_score) > abs(tech_score) and abs(sent_score) > abs(risk_score):
        horizon = "Medium-term"
        explanation.append("Sentiment dominant → medium-term focus")
    else:
        horizon = "Long-term"
        explanation.append("Risk/volume balanced → long-term horizon")

    final = FinalDecision(
        verdict=verdict,
        confidence_band=(round(low,3), round(high,3)),
        risk_adjusted_rating=round(risk_adj,3),
        investment_horizon=horizon,
        explanation_chain=explanation,
        conflict_notes=conflicts
    )

    return {
        "final_decision_obj": final,
        "live_weights": weights,
        "consensus_flag": len(conflicts) == 0,
        "step_timestamps": {**state.get("step_timestamps", {}), "supervisor": datetime.now().isoformat()}
    }

**Is code mein Supervisor Cross-Validation ho rahi hai, yani chaar agents (Technical, Sentiment, Risk, Volume) ke signals ko check karke final decision banana.**

Step-by-step short mein:

Data le li - tech, sentiment, risk, volume ke scores (objects se).

Conflicts detect kiye - agar 3 agents ek taraf aur 1 ulta, ya tech aur sentiment mein zyada difference ho.

Dynamic weighting - risk high ho ya sentiment extreme ho, toh unke weights badha diye (jaise risk badhne par risk weight 0.45 ho jata hai).

Composite score - sab scores ko normalized weights se multiply karke final score nikala.

Verdict decide - composite score ke hisaab se STRONG_BUY / BUY / HOLD / SELL / STRONG_SELL.

Confidence band - variance se margin nikal kar low-high range set ki.

Risk adjustment - conflicts ke count se composite score reduce kiya.

Investment horizon - dominant agent ke hisaab se Short/Medium/Long-term decide kiya.

Final object return - verdict, confidence band, risk-adjusted rating, horizon, explanation, conflicts sab bhej diya state mein.

Ek line mein: 4 signals ko weigh karke, conflicts resolve karte hue final buy/sell/hold decision aur confidence range produce karna.

In [13]:
# SECTION 10: GRAPH ROUTING LOGIC

def router_after_data(state: AnalystState) -> Literal["technical", "sentiment", "risk", "volume"]:
    return "technical"

def router_after_tech(state: AnalystState) -> Literal["sentiment", "risk", "volume", "supervisor"]:
    if state.get("sent_state_obj") is None: return "sentiment"
    if state.get("risk_metrics_obj") is None: return "risk"
    if state.get("vol_profile_obj") is None: return "volume"
    return "supervisor"

def router_after_sent(state: AnalystState) -> Literal["technical", "risk", "volume", "supervisor"]:
    if state.get("tech_signal_obj") is None: return "technical"
    if state.get("risk_metrics_obj") is None: return "risk"
    if state.get("vol_profile_obj") is None: return "volume"
    return "supervisor"

def router_after_risk(state: AnalystState) -> Literal["technical", "sentiment", "volume", "supervisor"]:
    if state.get("tech_signal_obj") is None: return "technical"
    if state.get("sent_state_obj") is None: return "sentiment"
    if state.get("vol_profile_obj") is None: return "volume"
    return "supervisor"

def router_after_vol(state: AnalystState) -> Literal["technical", "sentiment", "risk", "supervisor"]:
    if state.get("tech_signal_obj") is None: return "technical"
    if state.get("sent_state_obj") is None: return "sentiment"
    if state.get("risk_metrics_obj") is None: return "risk"
    return "supervisor"

def router_after_supervisor(state: AnalystState) -> Literal["__end__"]:
    return END

**Ye code Graph Routing Logic hai — matlab alag-alag "analyst" nodes (technical, sentiment, risk, volume) ko kis order me run karna hai, ye decide karta hai.**

Har analyst apna kaam karke state me ek object daalta hai (jaise tech_signal_obj, sent_state_obj, etc.)

Router functions check karte hain: Kaun sa object abhi bhi None hai? => uss analyst ke paas bhejo.

Jaise:

router_after_tech: agar sent_state_obj missing hai - "sentiment" pe jao, agar risk_metrics_obj missing - "risk" pe, aise.

Jab saare objects fill ho jate hain, toh sabse last me "supervisor" pe bhejta hai.

router_after_supervisor bas "__end__" return karta hai — matlab graph khatam.

Ek line me:
Jo bhi analyst ka kaam baaki hai, usko bulao; sab ho gaya toh supervisor ko bhejo, phir end.

In [14]:
# SECTION 11: BUILD LANGGRAPH WORKFLOW (WITH CUSTOM SERIALIZER)

def build_workflow() -> StateGraph:
    workflow = StateGraph(AnalystState)

    workflow.add_node("data_fetcher", lambda state: {"price_data": fetch_stock_data(state["ticker"], state["lookback"]), "step_timestamps": {"data_fetch": datetime.now().isoformat()}})
    workflow.add_node("technical", technical_expert_node)
    workflow.add_node("sentiment", sentiment_expert_node)
    workflow.add_node("risk", risk_expert_node)
    workflow.add_node("volume", volume_expert_node)
    workflow.add_node("supervisor", supervisor_node)

    workflow.add_edge(START, "data_fetcher")
    workflow.add_conditional_edges("data_fetcher", router_after_data, {"technical": "technical", "sentiment": "sentiment", "risk": "risk", "volume": "volume"})
    workflow.add_conditional_edges("technical", router_after_tech, {"sentiment":"sentiment","risk":"risk","volume":"volume","supervisor":"supervisor"})
    workflow.add_conditional_edges("sentiment", router_after_sent, {"technical":"technical","risk":"risk","volume":"volume","supervisor":"supervisor"})
    workflow.add_conditional_edges("risk", router_after_risk, {"technical":"technical","sentiment":"sentiment","volume":"volume","supervisor":"supervisor"})
    workflow.add_conditional_edges("volume", router_after_vol, {"technical":"technical","sentiment":"sentiment","risk":"risk","supervisor":"supervisor"})
    workflow.add_conditional_edges("supervisor", router_after_supervisor, {"__end__": END})

    # FIXED: Use custom serializer to handle DataFrame
    custom_serializer = CustomSerializer()
    checkpointer = InMemorySaver(serde=custom_serializer)
    return workflow.compile(checkpointer=checkpointer)

**Ye code ek LangGraph workflow bana raha hai jo stock market analysis ke liye multiple expert agents (technical, sentiment, risk, volume) ko ek supervisor ke control mein run karta hai.**

Short mein samjhe to:

Workflow banaya StateGraph(AnalystState) se - jaise ek pipeline jisme data flow hota hai.

Nodes add kiye - har node ek kaam karta hai:

data_fetcher - ticker aur lookback ke hisaab se stock data fetch karta hai.

technical, sentiment, risk, volume - alag-alag expert analysis karte hain.

supervisor - decide karta hai ki kaam khatam karna hai ya nahi.

Edges aur conditional routing:

START - data_fetcher

data_fetcher ke baad conditional routing: kis expert node pe jaana hai (technical, sentiment, risk, volume).

Har expert ke baad bhi conditional edges hain - ye decide karte hain ki agla kaun sa expert chale ya seedha supervisor ke paas jaye.

Supervisor ke baad, agar "__end__" condition milti hai, toh END (workflow khatam).

Checkpointer - InMemorySaver use kiya hai, lekin custom serializer diya hai. Kyunki price_data mein Pandas DataFrame ho sakta hai, jo normally serialise nahi hota. CustomSerializer DataFrame ko handle karne ke liye hai (jaise JSON mein convert karna).

Workflow compile karke return kar diya with checkpointer - isse workflow resume ho sakta hai (state save hota hai).

Ek line mein: Ye LangGraph workflow multiple expert agents ko dynamically route karta hai, aur DataFrame wale state ko save karne ke liye custom serializer lagata hai.

In [15]:
# SECTION 12: BILINGUAL VOICE GENERATION (English+Hindi)

def generate_bilingual_voice(verdict: str, horizon: str, risk_adj: float) -> str:
    """Create an English+Hindi mixed audio file using gTTS."""
    eng_text = f"Final analysis verdict is {verdict}. Recommended time horizon is {horizon}. Risk adjusted rating is {risk_adj} out of 2."
    hin_text = f"अंतिम विश्लेषण निर्णय {verdict} है। अनुशंसित समय क्षितिज {horizon} है। जोखिम समायोजित रेटिंग {risk_adj} है।"
    full_text = eng_text + " " + hin_text
    tts = gTTS(text=full_text, lang='hi', slow=False)  # Hindi TTS works for mixed due to Devanagari
    audio_path = "final_voice.mp3"
    tts.save(audio_path)
    return audio_path

**Ye code:**

Input: verdict, horizon, risk_adj (0-2)

English sentence banao: "Final analysis verdict is ..."

Hindi sentence banao: "अंतिम विश्लेषण निर्णय ..."

Dono ko mix karke full_text banao (English + Hindi)

gTTS (Google Text-to-Speech) ko lang='hi' pe chalao — ye Hindi accent me dono ko padhega (English words bhi Hindi style me)

Output: final_voice.mp3 file save karega

Return karega audio path? (code me return audio_path is galat likha hai, actually return audio_path hona chahiye)

Isme Ek mixed Hindi-English voice file generate hoti hai.

In [16]:
# SECTION 13: REPORT FORMATTER (Human readable)

def format_full_report(state: Dict) -> str:
    tech = state.get("tech_agent_raw", {})
    sent = state.get("sent_agent_raw", {})
    risk = state.get("risk_agent_raw", {})
    vol = state.get("vol_agent_raw", {})
    final = state.get("final_decision_obj")

    lines = []
    lines.append("\n" + "="*60)
    lines.append(f" MULTI-AGENT STOCK ANALYSIS REPORT")
    lines.append(f"Symbol: {state['ticker']} | Period: {state['lookback']}")
    lines.append(f"Report Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append("="*60)

    lines.append("\n TECHNICAL AGENT")
    lines.append(f"  Score: {tech.get('tech_score',0):.2f} | Signal: {tech.get('tech_signal','N/A')}")
    lines.append(f"  Stoch K: {tech.get('stoch_k','N/A')} | Williams%R: {tech.get('williams_r','N/A')}")

    lines.append("\n SENTIMENT AGENT")
    lines.append(f"  Verdict: {sent.get('verdict','N/A')} (Confidence: {sent.get('confidence',0):.2f})")
    lines.append(f"  Polarity: {sent.get('polarity',0):.3f} | Buzz: {sent.get('buzz_level',0):.2f}")

    lines.append("\n RISK AGENT")
    lines.append(f"  Risk Grade: {risk.get('risk_grade','N/A')} | Volatility: {risk.get('Volatility_annual',0):.1f}%")
    lines.append(f"  VaR: {risk.get('VaR_95',0):.2f}% | Max DD: {risk.get('Max_Drawdown',0):.2f}%")

    lines.append("\n VOLUME AGENT")
    lines.append(f"  Signal: {vol.get('vol_signal','N/A')} | Vol Ratio: {vol.get('vol_ratio',0):.2f}")
    lines.append(f"  Price-Vol Corr: {vol.get('price_vol_corr',0):.2f}")

    lines.append("\n" + "="*60)
    lines.append(" SUPERVISOR FINAL DECISION")
    if final:
        lines.append(f"  Verdict: {final.verdict}")
        lines.append(f"  Confidence Band: {final.confidence_band}")
        lines.append(f"  Risk-Adjusted Rating: {final.risk_adjusted_rating}")
        lines.append(f"  Horizon: {final.investment_horizon}")
        if final.explanation_chain:
            lines.append("  Explanation: " + "; ".join(final.explanation_chain[:3]))
        if final.conflict_notes:
            lines.append("  Conflicts: " + "; ".join(final.conflict_notes))
    else:
        lines.append("  Final decision not available.")
    lines.append("="*60)
    return "\n".join(lines)

**Ye code multi-agent stock analysis report ko human-readable format me print karta hai.**

Short me bole to:

Input: Ek dictionary state jisme different agents ke analysis results hote hain (technical, sentiment, risk, volume, aur final decision).

**Process:**

Pehle tech, sent, risk, vol agents ke raw data nikalta hai.

Phir ek formatted string banata hai with sections:

TECHNICAL AGENT: Score, signal, Stoch K, Williams%R

SENTIMENT AGENT: Verdict, confidence, polarity, buzz level

RISK AGENT: Risk grade, volatility, VaR, max drawdown

VOLUME AGENT: Signal, volume ratio, price-volume correlation

SUPERVISOR FINAL DECISION: Final verdict, confidence band, risk-adjusted rating, horizon, explanation, conflicts

Output: Ek neatly formatted report string jo console pe print kar sakte ho.

Matlab: Multi-agent system ne jo bhi analysis kiya (tech indicators, sentiment, risk, volume), supervisor ne final decision diya — ye function sabko ek saaf, line-by-line report me convert karta hai taaki human easily padh samajh sake.

In [17]:
# SECTION 14: MAIN EXECUTION (Async + Gradio)

async def run_multi_agent(symbol: str, period: str = "6mo") -> Dict:
    graph = build_workflow()
    initial_state: AnalystState = {
        "messages": [],
        "ticker": symbol.upper(),
        "lookback": period,
        "price_data": None,
        "tech_agent_raw": None,
        "sent_agent_raw": None,
        "risk_agent_raw": None,
        "vol_agent_raw": None,
        "tech_signal_obj": None,
        "sent_state_obj": None,
        "risk_metrics_obj": None,
        "vol_profile_obj": None,
        "final_decision_obj": None,
        "live_weights": None,
        "error_list": [],
        "step_timestamps": {},
        "consensus_flag": False
    }
    config = {"configurable": {"thread_id": f"{symbol}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"}}
    final_state = await graph.ainvoke(initial_state, config=config)
    return final_state

def gradio_interface(symbol: str, period: str):
    try:
        result = asyncio.run(run_multi_agent(symbol, period))
        report_text = format_full_report(result)
        final_obj = result.get("final_decision_obj")
        if final_obj:
            audio_file = generate_bilingual_voice(final_obj.verdict, final_obj.investment_horizon, final_obj.risk_adjusted_rating)
        else:
            audio_file = None
        return report_text, audio_file
    except Exception as e:
        return f"Error: {str(e)}", None

# Create Gradio app
with gr.Blocks(title="Multi-Agent Stock Analyzer with Bilingual Voice") as demo:
    gr.Markdown("# Multi-Agent LangGraph Stock Analysis + Bilingual Voice (English+Hindi)")
    with gr.Row():
        symbol_input = gr.Textbox(label="Stock Symbol", value="AAPL", placeholder="e.g., AAPL, MSFT, RELIANCE.NS")
        period_input = gr.Dropdown(label="Data Period", choices=["1mo","3mo","6mo","1y"], value="6mo")
    analyze_btn = gr.Button("Analyze")
    report_output = gr.Textbox(label="Analysis Report", lines=25)
    audio_output = gr.Audio(label="Bilingual Voice Output")
    analyze_btn.click(fn=gradio_interface, inputs=[symbol_input, period_input], outputs=[report_output, audio_output])

if __name__ == "__main__":
    demo.launch(share=True)

# SECTION 14: MAIN EXECUTION (Async + Gradio)

1. Gradio interface create hota hai – user chat kare ya input de

2. Async functions (async/await) use hote hain – jaise model se LLM response lena, taki UI freeze na ho

3. if __name__ == "__main__": block me sab start hota hai

4. gr.Blocks() ya gr.ChatInterface() ke saath queue() aur launch() call hota hai

5. (kabhi kabhi) asyncio.run(main()) likha hota hai agar sab async ho

Yeh section Gradio web UI ko async tareeke se chalaata hai, taake user input bhej sake aur background me model inference ho, bina UI atke.